# Gas Station Cost Efficiency Model Exploration

**Goal:** Demonstrate that the cheapest gas station per gallon is *not* always the most cost-efficient choice once driving distance and fuel consumption are factored in.

This notebook:
- Validates the core math model with concrete examples
- Identifies the break-even distance at which a cheaper station stops being worth the drive
- Shows how tank fill level changes the decision
- Builds a 2D cost surface to visualize the full decision space
- Ranks a synthetic set of stations the way the app does at runtime

In [ ]:
import sys, os

# Works whether Jupyter is launched from Gas App/ or Gas App/notebooks/
_cwd = os.getcwd()
_root = _cwd if os.path.exists(os.path.join(_cwd, 'costcalc.py')) else os.path.dirname(_cwd)
sys.path.insert(0, _root)
print(f'Project root: {_root}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from costcalc import VehicleParams, Station, calculate_station_result, rank_stations, recommend

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (10, 5)})
print('Setup complete.')

## The Math Model

For a vehicle with fuel efficiency **MPG** and **g_fill** gallons needed to reach a full tank:

$$\text{effective\_cost}_i = \left(g_{\text{fill}} + \frac{d_i}{\text{MPG}}\right) \times p_i$$

| Variable | Meaning |
|---|---|
| $d_i$ | Real road distance to station $i$ (miles) |
| $p_i$ | Price per gallon at station $i$ |
| $g_{\text{fill}}$ | `tank_capacity − tank_current` |

**Why this works:** Driving to a farther station burns fuel from your tank, which you must then replace at that station's price. So you buy more gallons there. The comparison is fair regardless of distance.

In [ ]:
# Concrete two-station example
vehicle = VehicleParams(mpg=28, tank_current=4.0, tank_capacity=13.0)
print(f'Vehicle: {vehicle.mpg} MPG | {vehicle.tank_current} gal in tank | {vehicle.gallons_to_fill:.1f} gal to fill')
print(f'Range remaining: {vehicle.range_remaining:.0f} miles\n')

station_a = Station('A — Close  ($4.00/gal, 1 mi)',  price_per_gallon=4.00, distance_miles=1.0)
station_b = Station('B — Far    ($3.50/gal, 12 mi)', price_per_gallon=3.50, distance_miles=12.0)

header = f"{'Station':<35} {'Price/gal':>10} {'Transit fuel':>14} {'Gal bought':>12} {'Eff. cost':>11}"
print(header)
print('-' * len(header))

for s in [station_a, station_b]:
    r = calculate_station_result(s, vehicle)
    print(f"{s.name:<35} ${s.price_per_gallon:>8.2f}  {r.fuel_burned_in_transit:>12.3f} gal {r.gallons_purchased:>10.3f} gal ${r.effective_cost:>9.2f}")

rec = recommend([station_a, station_b], vehicle)
print(f'\nWinner: {rec.best_station.station.name.strip()}')
print(f'Note:   {rec.note}')

## Break-Even Distance Analysis

Given a nearby baseline station, what is the **maximum distance** a cheaper station can be before the savings are wiped out by transit fuel cost?

Setting $\text{effective\_cost}_A = \text{effective\_cost}_B$ and solving for $d_B$:

$$d_{\text{break-even}} = \text{MPG} \times \left(\frac{(g_{\text{fill}} + d_A/\text{MPG}) \times p_A}{p_B} - g_{\text{fill}}\right)$$

In [ ]:
def break_even_distance(base_price, base_dist, comp_price, g_fill, mpg):
    """Max distance the cheaper station can be and still save money."""
    return mpg * ((g_fill + base_dist / mpg) * base_price / comp_price - g_fill)

base_price = 4.00
base_dist  = 0.5
mpg        = 28
g_fill     = 9.0  # mid-level tank

discounts = np.linspace(0.01, 0.90, 300)
comp_prices = base_price - discounts
be_dists = [break_even_distance(base_price, base_dist, p, g_fill, mpg) for p in comp_prices]

fig, ax = plt.subplots()
ax.plot(discounts, be_dists, lw=2.5, color='#1976D2')
ax.fill_between(discounts, be_dists, alpha=0.12, color='#1976D2')

# Annotate a few reference points
for disc, label in [(0.10, '10¢ off'), (0.25, '25¢ off'), (0.50, '50¢ off')]:
    d = break_even_distance(base_price, base_dist, base_price - disc, g_fill, mpg)
    ax.annotate(f'{label}\n→ drive up to {d:.1f} mi',
                xy=(disc, d), xytext=(disc + 0.05, d + 2),
                fontsize=8, arrowprops=dict(arrowstyle='->', color='gray'))

ax.set_xlabel('Price discount vs nearby station ($/gal)')
ax.set_ylabel('Max distance worth driving (miles)')
ax.set_title('Break-Even Distance — When Is It Worth the Drive?\n'
             f'(Baseline: ${base_price:.2f}/gal at {base_dist} mi | {mpg} MPG | {g_fill} gal to fill)')
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f mi'))
plt.tight_layout()
plt.show()

## Tank Fill Level Sensitivity

The **amount of gas you need to buy** ($g_{\text{fill}}$) is the dominant factor in whether price or distance matters more.

- **Tank nearly empty** → you're buying a lot of gas → price per gallon dominates → farther cheaper station often wins
- **Tank nearly full** → you're buying very little → transit fuel cost dominates → close station almost always wins

In [ ]:
base_price = 4.00
comp_price = 3.65
base_dist  = 0.5
tank_cap   = 13.0
mpg        = 28

scenarios = [
    ('Near empty (12 gal to fill)',  VehicleParams(mpg=mpg, tank_current=1.0,  tank_capacity=tank_cap), '#E53935'),
    ('Half tank (6.5 gal to fill)',  VehicleParams(mpg=mpg, tank_current=6.5,  tank_capacity=tank_cap), '#FB8C00'),
    ('Nearly full (1 gal to fill)',  VehicleParams(mpg=mpg, tank_current=12.0, tank_capacity=tank_cap), '#43A047'),
]

distances = np.linspace(0.2, 20, 400)
fig, ax = plt.subplots()

for label, veh, color in scenarios:
    savings = []
    baseline = Station('Base', price_per_gallon=base_price, distance_miles=base_dist)
    r_base = calculate_station_result(baseline, veh)
    for d in distances:
        if d / mpg > veh.tank_current:
            savings.append(np.nan)
        else:
            r = calculate_station_result(Station('Comp', price_per_gallon=comp_price, distance_miles=d), veh)
            savings.append(r_base.effective_cost - r.effective_cost)
    ax.plot(distances, savings, lw=2, label=label, color=color)

ax.axhline(0, color='black', lw=1, ls='--', label='Break-even')
ax.fill_between(distances, 0, 5, alpha=0.05, color='green')
ax.set_xlabel('Distance to the cheaper station (miles)')
ax.set_ylabel('Savings vs nearby expensive station ($)')
ax.set_title(f'Tank Fill Level Sensitivity\n'
             f'Nearby: ${base_price:.2f}/gal @ {base_dist} mi  vs  Farther: ${comp_price:.2f}/gal  |  {mpg} MPG')
ax.legend(title='Tank fill level')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))
ax.set_ylim(-3, None)
plt.tight_layout()
plt.show()

print('Observation: as the tank fills up, the break-even distance shrinks dramatically.')
print('A nearly-full tank almost never justifies driving more than a mile or two for cheaper gas.')

## Cost Surface Heatmap

The full decision space: for every combination of **(price, distance)** of a candidate station, is it better or worse than a fixed baseline nearby station?

- **Green** = candidate station saves money vs baseline
- **Red** = candidate station costs more overall
- **Black contour** = the break-even boundary

In [ ]:
vehicle  = VehicleParams(mpg=28, tank_current=4.0, tank_capacity=13.0)
baseline = Station('Baseline', price_per_gallon=4.00, distance_miles=0.5)
r_base   = calculate_station_result(baseline, vehicle)

prices    = np.linspace(3.00, 4.70, 80)
distances = np.linspace(0.1, 15, 80)
P, D = np.meshgrid(prices, distances)
savings_grid = np.full_like(P, np.nan)

for i, d in enumerate(distances):
    for j, p in enumerate(prices):
        if d / vehicle.mpg <= vehicle.tank_current:
            r = calculate_station_result(Station('x', price_per_gallon=p, distance_miles=d), vehicle)
            savings_grid[i, j] = r_base.effective_cost - r.effective_cost

fig, ax = plt.subplots(figsize=(11, 6))
vmax = np.nanmax(np.abs(savings_grid))
im = ax.contourf(P, D, savings_grid, levels=50, cmap='RdYlGn', vmin=-vmax, vmax=vmax)
ax.contour(P, D, savings_grid, levels=[0], colors='black', linewidths=2)

cbar = plt.colorbar(im, ax=ax)
cbar.set_label('Savings vs baseline station ($)', fontsize=10)

ax.scatter([4.00], [0.5], s=150, color='black', zorder=5, marker='*', label='Baseline station')
ax.set_xlabel('Price per gallon ($)', fontsize=11)
ax.set_ylabel('Distance to station (miles)', fontsize=11)
ax.set_title('Cost Surface: When Does a Cheaper Station Actually Save You Money?\n'
             '(Black line = break-even | Vehicle: 28 MPG, 9 gal to fill)', fontsize=12)
ax.legend(fontsize=9)
ax.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.0f mi'))
plt.tight_layout()
plt.show()

## Multi-Station Ranking

Simulate the app's recommendation engine across a realistic synthetic dataset — 15 stations with randomised prices and distances.

In [ ]:
np.random.seed(42)
NAMES = ['Shell', 'Chevron', 'Arco', '76', 'Valero', 'BP', 'Mobil', 'Texaco',
         'Sunoco', 'Marathon', 'Circle K', 'Kwik Trip', 'Casey\'s', 'QuikTrip', 'Wawa']

stations = [
    Station(
        name=NAMES[i],
        price_per_gallon=round(np.random.uniform(3.45, 4.55), 3),
        distance_miles=round(np.random.uniform(0.3, 12.0), 1),
    )
    for i in range(len(NAMES))
]

vehicle = VehicleParams(mpg=28, tank_current=4.0, tank_capacity=13.0)
ranked  = rank_stations(stations, vehicle)

df = pd.DataFrame([{
    'Rank':          r.rank,
    'Station':       r.station.name,
    'Price/gal':     r.station.price_per_gallon,
    'Distance (mi)': r.station.distance_miles,
    'Eff. Cost ($)': round(r.effective_cost, 2),
    'vs Best ($)':   round(r.savings_vs_best, 2),
} for r in ranked if r.reachable])

# --- Dual chart ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

colors = ['#43A047' if i == 0 else '#90CAF9' for i in range(len(df))]
bars = ax1.barh(df['Station'], df['Eff. Cost ($)'], color=colors)
ax1.set_xlabel('Effective Cost ($)')
ax1.set_title('Effective Cost by Station\n(green = recommended)')
ax1.invert_yaxis()
ax1.xaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0f'))
for bar, val in zip(bars, df['Eff. Cost ($)']):
    ax1.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
             f'${val:.2f}', va='center', fontsize=8)

sc = ax2.scatter(
    df['Distance (mi)'], df['Price/gal'],
    c=df['Eff. Cost ($)'], cmap='RdYlGn_r',
    s=120, zorder=3, edgecolors='white', linewidths=0.5
)
best = df.iloc[0]
ax2.scatter(best['Distance (mi)'], best['Price/gal'],
            s=300, color='gold', edgecolors='black', linewidths=1.5, zorder=5,
            marker='*', label=f'Recommended: {best["Station"]}')
plt.colorbar(sc, ax=ax2, label='Effective Cost ($)')
ax2.set_xlabel('Distance (miles)')
ax2.set_ylabel('Price per gallon ($)')
ax2.set_title('Price vs Distance\n(color = effective cost | ★ = recommended)')
ax2.legend(fontsize=9)
ax2.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.2f'))

plt.tight_layout()
plt.show()

print(df.to_string(index=False))